# Calling an Existing Agent

Shows how to retrieve and call an existing persistent Foundry v2 (Prompt) agent by its **name** (and optional version).
The same agent can serve multiple independent conversations — each `responses.create` call is stateless.

> **Before running:** create an agent with `01_persistent_agent.ipynb` (skip the cleanup cell),
> then set `AGENT_NAME` (and optionally `AGENT_VERSION`) below to the printed values.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
openai_client = client.get_openai_client()
print("Client ready.")

In [ ]:
# Set these to the name and version printed by 01_persistent_agent.ipynb
AGENT_NAME = "WriterAgent"
AGENT_VERSION = "1"

agent = client.agents.get_version(agent_name=AGENT_NAME, agent_version=AGENT_VERSION)
print(f"Using agent: {agent.name} (version {agent.version})")

In [ ]:
# Helper: send a prompt and stream the agent's response via the Responses API
def call_agent(prompt: str) -> None:
    stream = openai_client.responses.create(
        stream=True,
        input=prompt,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference",
            }
        },
    )

    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
    print()

In [ ]:
# First conversation
print("=== First Conversation ===\n")
call_agent("Write a poem about the sea.")

In [ ]:
# Second conversation — same agent, brand new thread
print("=== Second Conversation ===\n")
call_agent("Write a poem about the mountains.")